In [1]:
import pickle
import numpy as np
import torch

from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor, VecNormalize
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from sb3_contrib import RecurrentPPO

from rl_firm.env import make_sb_env

In [2]:
class RewardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []
    
    def _on_step(self) -> bool:
        # Check if episode ended
        if self.locals.get('dones', [False])[0]:
            # Get episode info from Monitor wrapper
            info = self.locals.get('infos', [{}])[0]
            if 'episode' in info:
                length = info['episode']['l']
                reward = info['episode']['r']
                self.episode_lengths.append(length)
                self.episode_rewards.append(reward)
                recent_avg_length = np.mean(self.episode_lengths[-20:])
                recent_avg = np.mean(self.episode_rewards[-20:])
                print(f"Episode {len(self.episode_rewards)}: Recent average episodic reward: {recent_avg:.2f}, recent average episodic length: {recent_avg_length:.2f}")
        return True

callback = RewardCallback()

In [3]:
vec_env = make_vec_env(make_sb_env, n_envs=8, vec_env_cls=SubprocVecEnv)
vec_env = VecMonitor(vec_env)

/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/hieul

In [5]:
%%time

# Initialize the PPO model
recurrent_ppo = RecurrentPPO(
    policy="MlpLstmPolicy",            
    env=vec_env,                            
    n_steps=1024,
    batch_size=2048,                               
    gamma=0.999,                  
    gae_lambda=0.95,             
    verbose=1,                    
    device="auto"               
)

Using cpu device
CPU times: user 7.31 ms, sys: 21.1 ms, total: 28.4 ms
Wall time: 73 ms


## 100k

In [6]:
%%time

recurrent_ppo.learn(total_timesteps=100000, callback=callback, reset_num_timesteps=False)

Episode 1: Recent average episodic reward: 0.11, recent average episodic length: 17.00
Episode 2: Recent average episodic reward: 0.08, recent average episodic length: 19.00
Episode 3: Recent average episodic reward: 0.08, recent average episodic length: 16.67
Episode 4: Recent average episodic reward: 0.28, recent average episodic length: 23.00
Episode 5: Recent average episodic reward: 0.22, recent average episodic length: 19.60
Episode 6: Recent average episodic reward: 0.19, recent average episodic length: 18.17
Episode 7: Recent average episodic reward: 0.17, recent average episodic length: 16.00
Episode 8: Recent average episodic reward: 0.16, recent average episodic length: 15.75
Episode 9: Recent average episodic reward: 0.15, recent average episodic length: 17.11
Episode 10: Recent average episodic reward: 0.16, recent average episodic length: 18.90
Episode 11: Recent average episodic reward: 0.15, recent average episodic length: 18.55
Episode 12: Recent average episodic rewar

In [7]:
recurrent_ppo.save('results/model/sb3_contrib/100k')

/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/stable_baselines3/common/save_util.py:284: UserWarning: Path 'results/model/sb3_contrib' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


## 1m

In [6]:
%%time

recurrent_ppo.learn(total_timesteps=1000000, callback=callback, reset_num_timesteps=False)

Episode 1: Recent average episodic reward: 0.08, recent average episodic length: 6.00
Episode 2: Recent average episodic reward: 0.11, recent average episodic length: 8.00
Episode 3: Recent average episodic reward: 0.11, recent average episodic length: 11.67
Episode 4: Recent average episodic reward: 0.11, recent average episodic length: 11.75
Episode 5: Recent average episodic reward: 0.09, recent average episodic length: 10.40
Episode 6: Recent average episodic reward: 0.08, recent average episodic length: 10.50
Episode 7: Recent average episodic reward: 0.07, recent average episodic length: 10.14
Episode 8: Recent average episodic reward: 0.06, recent average episodic length: 9.00
Episode 9: Recent average episodic reward: 0.07, recent average episodic length: 10.56
Episode 10: Recent average episodic reward: 0.06, recent average episodic length: 9.70
Episode 11: Recent average episodic reward: 0.07, recent average episodic length: 11.82
Episode 12: Recent average episodic reward: 0

In [7]:
recurrent_ppo.save('results/model/sb3_contrib/1m')

## 2m

In [8]:
%%time

recurrent_ppo.learn(total_timesteps=1000000, callback=callback, reset_num_timesteps=False)

Episode 2102: Recent average episodic reward: 3.72, recent average episodic length: 176.45
Episode 2103: Recent average episodic reward: 3.93, recent average episodic length: 186.75
Episode 2104: Recent average episodic reward: 3.66, recent average episodic length: 180.95
Episode 2105: Recent average episodic reward: 3.55, recent average episodic length: 180.55
Episode 2106: Recent average episodic reward: 3.54, recent average episodic length: 179.20
Episode 2107: Recent average episodic reward: 3.52, recent average episodic length: 177.50
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 145      |
|    ep_rew_mean     | 2.84     |
| time/              |          |
|    fps             | 36       |
|    iterations      | 1        |
|    time_elapsed    | 223      |
|    total_timesteps | 1015808  |
---------------------------------
Episode 2108: Recent average episodic reward: 3.51, recent average episodic length: 171.80
Episode 2109: Recent av

In [9]:
recurrent_ppo.save('results/model/sb3_contrib/2m')

## 3m

In [10]:
%%time

recurrent_ppo.learn(total_timesteps=1000000, callback=callback, reset_num_timesteps=False)

Episode 2862: Recent average episodic reward: 8.69, recent average episodic length: 253.00
Episode 2863: Recent average episodic reward: 8.73, recent average episodic length: 256.25
Episode 2864: Recent average episodic reward: 8.21, recent average episodic length: 239.30
Episode 2865: Recent average episodic reward: 8.21, recent average episodic length: 239.30
Episode 2866: Recent average episodic reward: 7.52, recent average episodic length: 221.10
Episode 2867: Recent average episodic reward: 7.44, recent average episodic length: 222.45
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 217      |
|    ep_rew_mean     | 7.31     |
| time/              |          |
|    fps             | 30       |
|    iterations      | 1        |
|    time_elapsed    | 271      |
|    total_timesteps | 2023424  |
---------------------------------
Episode 2868: Recent average episodic reward: 7.10, recent average episodic length: 224.40
Episode 2869: Recent av

In [11]:
recurrent_ppo.save('results/model/sb3_contrib/3m')

## 4m

In [12]:
%%time

recurrent_ppo.learn(total_timesteps=1000000, callback=callback, reset_num_timesteps=False)

Episode 3465: Recent average episodic reward: 7.14, recent average episodic length: 224.55
Episode 3466: Recent average episodic reward: 7.14, recent average episodic length: 224.65
Episode 3467: Recent average episodic reward: 7.61, recent average episodic length: 233.40
Episode 3468: Recent average episodic reward: 7.00, recent average episodic length: 217.50
Episode 3469: Recent average episodic reward: 7.41, recent average episodic length: 229.00
Episode 3470: Recent average episodic reward: 7.60, recent average episodic length: 226.85
Episode 3471: Recent average episodic reward: 7.21, recent average episodic length: 219.95
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 245      |
|    ep_rew_mean     | 8.62     |
| time/              |          |
|    fps             | 39       |
|    iterations      | 1        |
|    time_elapsed    | 209      |
|    total_timesteps | 3031040  |
---------------------------------
Episode 3472: Recent av

In [13]:
recurrent_ppo.save('results/model/sb3_contrib/4m')